<a href="https://colab.research.google.com/github/huamanchristian44/LAB07--HE/blob/develop/Laboratorio07_Mineria_de_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Laboratorio 7: Regresión logística. Máquinas de vector de soporte**

a. Calcule el information value (IV), tanto para el grupo de variable numéricas como categóricas y excluya las que tenga un poder predictivo débil o menor. Además, separe la variable de clasificación del resto de variables para luego obtener los datos de entrenamiento y prueba, tomando de este último el 25% de datos.

In [116]:
#Importando librerías
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [117]:
#Cargando los datos desde la URL
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
columnas = ['id', 'clump_thickness', 'uniformity_cell_size', 'uniformity_cell_shape',
            'marginal_adhesion', 'single_epithelial_size', 'bare_nuclei',
            'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class']

df = pd.read_csv(url, names=columnas)
df.head(5)

,id,clump_thickness,uniformity_cell_size,uniformity_cell_shape,marginal_adhesion,single_epithelial_size,bare_nuclei,bland_chromatin,normal_nucleoli,mitoses,class
0,1000025,5,1,1,1,2,1,3,1,1,2
1,1002945,5,4,4,5,7,10,3,2,1,2
2,1015425,3,1,1,1,2,2,3,1,1,2
3,1016277,6,8,8,1,3,4,3,7,1,2
4,1017023,4,1,1,3,2,1,3,1,1,2


In [118]:
#Reemplazando '?' por valores nulos en todo el DataFrame
df.replace('?', pd.NA, inplace=True)

In [119]:
#Convertiendo la columna 'bare_nuclei' a tipo numérico (esto convertirá los NA automáticamente)
df['bare_nuclei'] = pd.to_numeric(df['bare_nuclei'], errors='coerce')

In [120]:
#Reemplazando los valores faltantes por la mediana de la columna
mediana = df['bare_nuclei'].median()
df['bare_nuclei'].fillna(mediana, inplace=True)

<ipython-input-120-c5cbbf1e5595>:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['bare_nuclei'].fillna(mediana, inplace=True)


In [137]:
#Asegurandonos de que la columna sea entera
df['bare_nuclei'] = df['bare_nuclei'].astype(int)

In [122]:
#Convertiendo la clase: 2 = Benigno (0), 4 = Maligno (1)
df['class'] = df['class'].map({2: 0, 4: 1})

In [123]:
#Cálculo de WOE e IV
def calc_woe_iv(df, feature, target):
    lst = []
    for i in range(df[feature].nunique()):
        val = df[feature].unique()[i]
        total = df[df[feature] == val].count()[feature]
        good = df[(df[feature] == val) & (df[target] == 0)].count()[feature]
        bad = df[(df[feature] == val) & (df[target] == 1)].count()[feature]
        dist_good = good / df[df[target] == 0].count()[feature]
        dist_bad = bad / df[df[target] == 1].count()[feature]
        woe = np.log(dist_good / dist_bad) if dist_bad != 0 and dist_good != 0 else 0
        iv = (dist_good - dist_bad) * woe
        lst.append(iv)
    return sum(lst)

In [124]:
#IV para todas las variables
iv_dict = {}
for col in df.columns:
    if col not in ['id', 'class']:
        iv = calc_woe_iv(df, col, 'class')
        iv_dict[col] = iv

In [125]:
#Visualizando los resultados
iv_df = pd.DataFrame.from_dict(iv_dict, orient='index', columns=['IV']).sort_values(by='IV', ascending=False)
print(iv_df)

                              IV
uniformity_cell_shape   5.166413
bare_nuclei             4.799511
uniformity_cell_size    4.518824
single_epithelial_size  3.940390
bland_chromatin         3.433822
marginal_adhesion       2.872259
normal_nucleoli         2.443920
clump_thickness         2.303649
mitoses                 1.023807


In [126]:
#Eliminando las variables con IV < 0.02
selected_features = iv_df[iv_df['IV'] >= 0.02].index.tolist()
X = df[selected_features]
y = df['class']

In [127]:
#Separando en entrenamiento (75%) y prueba (25%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

b. Genere el modelo de regresión logística y evalúe la exclusión de variables mediante la
significancia de los coeficientes. Además, calcule las métricas de clasificación que se
implementaron en la parte práctica e interprete sus resultados más importantes.


In [128]:
#Entrenando el modelo
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [129]:
#Visualizando coeficientes
coef_df = pd.DataFrame({
    'Feature': selected_features,
    'Coefficient': lr.coef_[0]
})
print(coef_df)

                  Feature  Coefficient
0   uniformity_cell_shape     0.428076
1             bare_nuclei     0.432242
2    uniformity_cell_size    -0.021021
3  single_epithelial_size     0.140046
4         bland_chromatin     0.393740
5       marginal_adhesion     0.191906
6         normal_nucleoli     0.009416
7         clump_thickness     0.515303
8                 mitoses     0.344237


In [130]:
#Predicción
y_pred_lr = lr.predict(X_test)

In [131]:
#Métricas
print("Métricas de Regresión Logística:")
print(classification_report(y_test, y_pred_lr))
print("Matriz de Confusión:\n", confusion_matrix(y_test, y_pred_lr))

Métricas de Regresión Logística:
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       118
           1       0.96      0.91      0.94        57

    accuracy                           0.96       175
   macro avg       0.96      0.95      0.95       175
weighted avg       0.96      0.96      0.96       175

Matriz de Confusión:
 [[116   2]
 [  5  52]]


c. Genere el modelo SVM, calcule sus métricas de clasificación y compárelas con las del modelo
de regresión logística para ver si hubo o no mejoras.

In [132]:
from sklearn.svm import SVC

In [133]:
#Entrenando modelo SVM
svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)

SVC(kernel='linear')

In [134]:
#Predicción
y_pred_svm = svm_model.predict(X_test)

In [135]:
#Métricas
print("Métricas de SVM:")
print(classification_report(y_test, y_pred_svm))
print("Matriz de Confusión:\n", confusion_matrix(y_test, y_pred_svm))

Métricas de SVM:
              precision    recall  f1-score   support

           0       0.97      0.98      0.97       118
           1       0.96      0.93      0.95        57

    accuracy                           0.97       175
   macro avg       0.97      0.96      0.96       175
weighted avg       0.97      0.97      0.97       175

Matriz de Confusión:
 [[116   2]
 [  4  53]]
